In [1]:
import pandas as pd, numpy as np
import json, os

In [2]:
from jh_utils.time_series.time_dataframe import time_series_dataframe
from jh_utils.pandas.preprocessing import make_dummies

In [3]:
## read dataset
df = pd.read_csv('../../data/03_processed/processed.csv')

## Na from dataset
na_value = -9999

## organize index and drop hour and dates
df['date_time'] = df['data']+' '+df['hora']
df['date_time'] = pd.to_datetime(df['date_time'])

## adding week, month and year variables
df['weekday'] = df['date_time'].dt.weekday
df['month'] = df['date_time'].dt.month
# df['year'] = df['date_time'].dt.year

df = pd.concat((df,make_dummies(df['weekday'])),axis=1)
df = pd.concat((df,make_dummies(df['month'])),axis=1)

In [10]:
df = df.rename(columns={'hora': 'hour'})
df.hour = df.hour.apply(lambda x: x.split(':')[0])
df = pd.concat((df,make_dummies(df['hour'])),axis=1)

In [11]:
## set an index
df.index = df['date_time']
df = df.drop(columns=['data','date_time','hour','weekday','month'])
df = df.replace(to_replace=na_value,value=np.NaN)

## fill na with previous values
df = df.fillna(method='ffill')

## variables to be used
rows    = df.shape[0]
columns = df.shape[1]

In [88]:
df1 = time_series_dataframe('2006-10-25','2021-01-01')

In [95]:
df = df[df.date_time>'2006-10-31']

In [96]:
df

,date_time,"A612 - precipitacao total, horario (mm)","A612 - pressao atmosferica ao nivel da estacao, horaria (mb)",A612 - pressao atmosferica max. na hora ant. (aut) (mb),A612 - pressao atmosferica min. na hora ant. (aut) (mb),"A612 - temperatura do ar - bulbo seco, horaria (°c)",A612 - temperatura do ponto de orvalho (°c),A612 - temperatura maxima na hora ant. (aut) (°c),A612 - temperatura minima na hora ant. (aut) (°c),A612 - temperatura orvalho max. na hora ant. (aut) (°c),...,hour_14,hour_15,hour_16,hour_17,hour_18,hour_19,hour_20,hour_21,hour_22,hour_23
0,2006-10-31 16:00:00,0.0,1010.9,1011.8,1010.9,27.4,22.0,27.7,26.2,22.5,...,0,0,1,0,0,0,0,0,0,0
1,2006-10-31 17:00:00,1.0,1010.6,1011.0,1010.5,26.0,20.5,27.9,26.0,22.3,...,0,0,0,1,0,0,0,0,0,0
2,2006-10-31 18:00:00,0.0,1010.3,1010.6,1010.2,26.3,22.3,26.8,25.3,22.3,...,0,0,0,0,1,0,0,0,0,0
3,2006-10-31 19:00:00,0.0,1009.8,1010.3,1009.7,26.0,21.7,27.0,26.0,22.2,...,0,0,0,0,0,1,0,0,0,0
4,2006-10-31 20:00:00,0.0,1010.3,1010.4,1009.7,25.2,22.0,26.3,25.2,22.0,...,0,0,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124347,2020-12-31 19:00:00,0.0,1011.6,1012.1,1011.6,29.5,16.2,30.6,29.4,17.5,...,0,0,0,0,0,1,0,0,0,0
124348,2020-12-31 20:00:00,0.0,1011.4,1011.7,1011.4,28.1,17.8,29.6,28.1,17.9,...,0,0,0,0,0,0,1,0,0,0
124349,2020-12-31 21:00:00,0.0,1012.3,1012.3,1011.4,26.6,18.8,28.1,26.6,18.8,...,0,0,0,0,0,0,0,1,0,0
124350,2020-12-31 22:00:00,0.0,1013.0,1013.0,1012.3,25.8,19.7,26.6,25.8,19.7,...,0,0,0,0,0,0,0,0,1,0


In [12]:
datasets = '../config/datasets.json'
with open(datasets) as data:
    datasets = json.load(data)

dataset_keys = list(datasets.keys())

FileNotFoundError: [Errno 2] No such file or directory: '../config/datasets.json'

In [97]:
def make_model_dataframe(df, dataset, fixed_columns = ['hour', 'month', 'weekday']):
    ## filtering the date
    df = df[df.index > datasets[dataset]['start_date']]
    ## declaring the output-dataframe
    df_out = pd.DataFrame()
    ## filtering regex
    for i in datasets[dataset]['regex_columns'] + fixed_columns:
        df_out = pd.concat([df_out,df.filter(regex=i)],axis=1)
    return df_out

In [103]:
for i in dataset_keys:
    make_model_dataframe(df,i).to_csv('../data/datasets/{dataset}.csv'.format(dataset = i))